In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [2]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [3]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [5]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [8]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [9]:
import numpy as np
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver
import os
from tqdm import tqdm

# ==============================================================================
# 1. [표 적용] 파라미터 설정
# ==============================================================================
# Table Parameter Mapping
CARRIER_FREQUENCY = 5.9e9  # 5.9 GHz
BANDWIDTH = 15e6           # 15 MHz
TX_POWER_DBM = 26.0        # 26 dBm
SAMPLING_INTERVAL = 0.0005 # 0.5 ms (요청하신 값)

# 장면 주파수 설정 (재질 특성 계산에 중요)
scene.frequency = CARRIER_FREQUENCY

# ==============================================================================
# 2. 이동 경로 및 시간 설정
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

# 경로 인덱스 (기존과 동일)
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 

# 거리 계산
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]

# 속도 설정: 60 km/h
speed_ms = 60.0 / 3.6 
total_time_full = total_distance / speed_ms

# ------------------------------------------------------------------------------
# ⚠️ [중요] 0.5ms 간격은 데이터가 매우 많으므로 테스트를 위해 시간을 제한합니다.
# 전체 경로를 다 돌리려면 아래 limit_time을 total_time_full로 변경하세요.
# ------------------------------------------------------------------------------
limit_time = 1.0  # 테스트용: 1초만 시뮬레이션 (약 2000 프레임)
# limit_time = total_time_full # 전체 경로 시뮬레이션 (시간 매우 오래 걸림 주의)

time_steps = np.arange(0, limit_time, SAMPLING_INTERVAL)

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    seg_start, seg_len = cumulative_dists[idx], segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    ratio = (target_dist - seg_start) / seg_len
    return waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])

# ==============================================================================
# 3. Tx/Rx 재배치 (파라미터 적용)
# ==============================================================================
# 기존 객체 제거
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

# 안테나 설정 (VH 편파 등은 유지)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

# Tx 위치 (기존 좌표 유지)
tx_positions = [[-125.66, 56.36, -181.45], [323.47, 36.87, -204.31], [0.66, 56.36, -181.45]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    # [적용] power_dbm=26.0
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=TX_POWER_DBM)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

# Rx 초기화
rx = Receiver(name="rx_car", position=get_pos_at_time(0.0))
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 4. 고해상도 시뮬레이션 실행 (0.5ms 단위)
# ==============================================================================
history_a = []      
history_tau = []    
history_pos = []    
history_time = []   

print(f"🚀 고해상도 시뮬레이션 시작")
print(f" - 주파수: {CARRIER_FREQUENCY/1e9} GHz")
print(f" - 송신 전력: {TX_POWER_DBM} dBm")
print(f" - 관측 간격: {SAMPLING_INTERVAL*1000} ms")
print(f" - 총 프레임 수: {len(time_steps)}")

for t in tqdm(time_steps):
    pos = get_pos_at_time(t)
    
    rx.position = pos
    # Tx가 차를 바라보도록 업데이트 (선택 사항, 고정형이면 제거 가능)
    for name in tx_names:
        scene.transmitters[name].look_at(pos)
        
    # 경로 계산
    paths = solver(scene, max_depth=3, samples_per_src=100000, 
                   diffuse_reflection=True, diffraction=True)
    
    # 데이터 추출
    if paths.a is not None:
        try:
            a_val = np.array(paths.a)    
            tau_val = np.array(paths.tau)
        except:
            a_val = np.array([])
            tau_val = np.array([])
        history_a.append(a_val)
        history_tau.append(tau_val)
    else:
        history_a.append(np.array([]))
        history_tau.append(np.array([]))
        
    history_pos.append(pos)
    history_time.append(t)

# ==============================================================================
# 5. 저장
# ==============================================================================
num_frames = len(history_a)
a_objects = np.empty(num_frames, dtype=object)
tau_objects = np.empty(num_frames, dtype=object)

for i in range(num_frames):
    a_objects[i] = history_a[i]
    tau_objects[i] = history_tau[i]

save_path = 'channel_history_high_res.npz'
np.savez(save_path, 
         a=a_objects, 
         tau=tau_objects,
         pos=np.array(history_pos),
         time=np.array(history_time),
         params={
             'freq': CARRIER_FREQUENCY,
             'bw': BANDWIDTH,
             'power': TX_POWER_DBM,
             'interval': SAMPLING_INTERVAL
         })

print(f"\n💾 저장 완료: {save_path}")

🚀 고해상도 시뮬레이션 시작
 - 주파수: 5.9 GHz
 - 송신 전력: 26.0 dBm
 - 관측 간격: 0.5 ms
 - 총 프레임 수: 2000


100%|██████████| 2000/2000 [23:59<00:00,  1.39it/s]


💾 저장 완료: channel_history_high_res.npz


In [ ]:
import numpy as np
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver
import os
from tqdm import tqdm

# ==============================================================================
# 1. [표 적용] 파라미터 설정 (Table Values)
# ==============================================================================
CARRIER_FREQUENCY = 5.9e9     # 5.9 GHz
BANDWIDTH = 15e6              # 15 MHz
TX_POWER_DBM = 26.0           # 26 dBm

# [핵심] 버스트 구조 설정
BURST_RATE = 20.0             # 20 Hz (초당 20번 버스트)
BURST_INTERVAL = 1.0 / BURST_RATE  # = 50 ms (0.05s)
SNAPSHOTS_PER_BURST = 30      # 버스트 당 30개 스냅샷
BURST_DURATION = 640e-6       # 640 us (0.00064s)
# 스냅샷 간 시간 간격 = 640us / 30 ≈ 21.3 us
SNAPSHOT_INTERVAL = BURST_DURATION / SNAPSHOTS_PER_BURST 

# 장면 설정
scene.frequency = CARRIER_FREQUENCY
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

# ==============================================================================
# 2. Tx/Rx 배치
# ==============================================================================
# 기존 객체 제거
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

tx_positions = [[-125.66, 56.36, -181.45], [323.47, 36.87, -204.31], [0.66, 56.36, -181.45]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=TX_POWER_DBM)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

# 초기 Rx 생성
rx = Receiver(name="rx_car", position=[0,0,0])
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 3. 경로 데이터 준비
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

# (기존 경로 로직 유지)
path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]
speed_ms = 60.0 / 3.6 
total_time_full = total_distance / speed_ms

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    seg_start, seg_len = cumulative_dists[idx], segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    ratio = (target_dist - seg_start) / seg_len
    return waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])

# ==============================================================================
# 4. [버스트 구조] 시뮬레이션 실행
# ==============================================================================
# 예시: 1초(20개의 버스트)만 시뮬레이션
# (전체 경로를 다 하려면 SIM_DURATION = total_time_full 로 설정하세요)
SIM_DURATION = 1.0 
num_bursts = int(SIM_DURATION * BURST_RATE)

print(f"🚀 버스트 시뮬레이션 시작 (Table Spec)")
print(f" - 총 버스트 수: {num_bursts} (Rate: {BURST_RATE} Hz)")
print(f" - 버스트 당 스냅샷: {SNAPSHOTS_PER_BURST} (Duration: {BURST_DURATION*1e6:.1f} us)")
print(f" - 총 생성될 데이터 프레임: {num_bursts * SNAPSHOTS_PER_BURST}")

# 결과를 저장할 리스트 (Burst 단위로 묶어서 저장 추천)
# 구조: [Burst_0_data, Burst_1_data, ...]
burst_data_list = []

for b_idx in tqdm(range(num_bursts), desc="Bursts"):
    burst_start_time = b_idx * BURST_INTERVAL
    
    # 한 버스트 내의 30개 스냅샷 데이터
    snapshots_a = []
    snapshots_tau = []
    snapshots_pos = []
    snapshots_time = []
    
    for s_idx in range(SNAPSHOTS_PER_BURST):
        # 현재 스냅샷의 절대 시간
        current_time = burst_start_time + (s_idx * SNAPSHOT_INTERVAL)
        
        # 1. 위치 업데이트
        pos = get_pos_at_time(current_time)
        rx.position = pos
        for name in tx_names:
            scene.transmitters[name].look_at(pos) # (옵션)
            
        # 2. 경로 계산
        paths = solver(scene, max_depth=3, samples_per_src=100000, 
                       diffuse_reflection=True, diffraction=True)
        
        # 3. 데이터 추출
        if paths.a is not None:
            snapshots_a.append(np.array(paths.a))
            snapshots_tau.append(np.array(paths.tau))
        else:
            snapshots_a.append(np.array([]))
            snapshots_tau.append(np.array([]))
            
        snapshots_pos.append(pos)
        snapshots_time.append(current_time)
    
    # 버스트 하나가 끝나면 묶어서 저장
    burst_data_list.append({
        'burst_index': b_idx,
        'a': snapshots_a,       # (30, ...)
        'tau': snapshots_tau,   # (30, ...)
        'pos': snapshots_pos,   # (30, 3)
        'time': snapshots_time  # (30,)
    })

# ==============================================================================
# 5. 저장 (구조화된 파일)
# ==============================================================================
save_path = 'channel_history_burst.npy' # 리스트 저장을 위해 .npy 사용 (pickle)
np.save(save_path, burst_data_list)

print(f"\n💾 저장 완료: {save_path}")
print(f"데이터는 {len(burst_data_list)}개의 버스트로 구성되어 있으며,")
print(f"각 버스트는 {SNAPSHOTS_PER_BURST}개의 연속된 스냅샷을 포함합니다.")

🚀 버스트 시뮬레이션 시작 (Table Spec)
 - 총 버스트 수: 20 (Rate: 20.0 Hz)
 - 버스트 당 스냅샷: 30 (Duration: 640.0 us)
 - 총 생성될 데이터 프레임: 600


Bursts:   5%|▌         | 1/20 [00:22<07:03, 22.29s/it]